# QRRP — Quantum RAG Rating Prediction (MovieLens)
Notebook complet pour générer des résultats **LLM**, **Quantum kNN**, **Grover-only LLM**, **Q-LLM (Hybrid)**.

**Remarque**: ce notebook est auto‑contenu mais suppose que tu as accès aux fichiers MovieLens (ratings.csv, movies.csv) localement.


In [10]:
!kill -9 2729162


In [11]:
!nvidia-smi

Wed Dec 31 20:22:43 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.124.06             Driver Version: 570.124.06     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L40S                    Off |   00000000:E1:00.0 Off |                    0 |
| N/A   33C    P0             76W /  350W |    2787MiB /  46068MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [1]:

# =========================
# 0) Imports & config
# =========================
import os, re, json, math, random
import numpy as np
import pandas as pd

from dataclasses import dataclass
from typing import Dict, Tuple, Optional, List

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# -------- Paths MovieLens --------
MOVIELENS_DIR = os.environ.get("MOVIELENS_DIR", "./ml-latest-small")
RATINGS_PATH = os.path.join(MOVIELENS_DIR, "ratings.csv")
MOVIES_PATH  = os.path.join(MOVIELENS_DIR, "movies.csv")

# -------- Evaluation --------
TEST_SIZE = 300
K_NEIGHBORS = 8
SHOTS = 400
GROVER_ITERS = 1

# -------- LLM --------
DEVICE = "cuda" if os.environ.get("CUDA_VISIBLE_DEVICES", "") not in ["", "-1"] else "cpu"
LLM_MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

print("DEVICE:", DEVICE)
print("Paths:", RATINGS_PATH, MOVIES_PATH)


DEVICE: cuda
Paths: ./ml-latest-small/ratings.csv ./ml-latest-small/movies.csv


In [2]:

# =========================
# 1) Load MovieLens
# =========================
assert os.path.exists(RATINGS_PATH), f"ratings.csv introuvable: {RATINGS_PATH}"
assert os.path.exists(MOVIES_PATH),  f"movies.csv introuvable: {MOVIES_PATH}"

ratings = pd.read_csv(RATINGS_PATH)
movies  = pd.read_csv(MOVIES_PATH)

# MovieLens ratings: float. Pour rester sur 1..5 entiers:
ratings["rating_int"] = ratings["rating"].round().clip(1,5).astype(int)

ratings_df = ratings.merge(movies, on="movieId", how="left")
ratings_df = ratings_df[["userId","movieId","rating_int","timestamp","title","genres"]].rename(columns={"rating_int":"rating"})

print("ratings_df shape:", ratings_df.shape)
print("ratings_df columns:", ratings_df.columns.tolist())
ratings_df.head()


ratings_df shape: (100836, 6)
ratings_df columns: ['userId', 'movieId', 'rating', 'timestamp', 'title', 'genres']


,userId,movieId,rating,timestamp,title,genres
0,1,1,4,964982703,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,1,3,4,964981247,Grumpier Old Men (1995),Comedy|Romance
2,1,6,4,964982224,Heat (1995),Action|Crime|Thriller
3,1,47,5,964983815,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
4,1,50,5,964982931,"Usual Suspects, The (1995)",Crime|Mystery|Thriller


In [3]:

# =========================
# 2) Train/Test split
# =========================
df = ratings_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
test_df  = df.iloc[:TEST_SIZE].copy()
train_df = df.iloc[TEST_SIZE:].copy()
print("train:", train_df.shape, "test:", test_df.shape)


train: (100536, 6) test: (300, 6)


In [4]:
# *********************************** Cellule 3 — Retrieval kNN ************************
# =========================
# 3) TF-IDF item encoder + neighbor retrieval (RAG)
# =========================
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

movie_meta = ratings_df[["movieId","title","genres"]].drop_duplicates("movieId").copy()
movie_meta["text"] = (movie_meta["title"].fillna("") + " | " + movie_meta["genres"].fillna("")).astype(str)

vectorizer = TfidfVectorizer(max_features=20000, ngram_range=(1,2))
X_items = vectorizer.fit_transform(movie_meta["text"].tolist())
movieid_to_index = {mid:i for i, mid in enumerate(movie_meta["movieId"].tolist())}

def retrieve_neighbors(user_id: int, target_movie_id: int, ratings: pd.DataFrame, k: int) -> pd.DataFrame:
    hist = ratings[(ratings["userId"]==user_id) & (ratings["movieId"]!=target_movie_id)]
    if hist.empty or target_movie_id not in movieid_to_index:
        return hist.head(0)

    t_idx = movieid_to_index[target_movie_id]
    hist_movie_ids = hist["movieId"].tolist()
    hist_idx = [movieid_to_index[m] for m in hist_movie_ids if m in movieid_to_index]
    if len(hist_idx)==0:
        return hist.head(0)

    sims = cosine_similarity(X_items[t_idx], X_items[hist_idx]).flatten()
    order = np.argsort(-sims)[:k]
    picked = hist.iloc[order].copy()
    picked["sim"] = sims[order]

    # éviter title_x/title_y: drop puis merge propre
    picked = picked.drop(columns=[c for c in ["title","genres"] if c in picked.columns], errors="ignore")
    picked = picked.merge(movie_meta[["movieId","title","genres"]], on="movieId", how="left")
    return picked.sort_values("sim", ascending=False).reset_index(drop=True)

def format_rag_context(neigh: pd.DataFrame) -> str:
    if neigh is None or len(neigh)==0:
        return "No user history available."
    lines = []
    for _, row in neigh.iterrows():
        lines.append(
            "- {t} ({g}): user_rating={r}, sim={s:.3f}".format(
                t=row.get("title","<unk>"),
                g=row.get("genres","<unk>"),
                r=row.get("rating", None),
                s=float(row.get("sim", 0.0))
            )
        )
    return "\n".join(lines)

sample_user = int(test_df["userId"].iloc[0])
sample_movie = int(test_df["movieId"].iloc[0])
neigh = retrieve_neighbors(sample_user, sample_movie, train_df, K_NEIGHBORS)
print("neigh columns:", neigh.columns.tolist())
print(format_rag_context(neigh)[:800])


neigh columns: ['userId', 'movieId', 'rating', 'timestamp', 'sim', 'title', 'genres']
- Troy (2004) (Action|Adventure|Drama|War): user_rating=4, sim=0.226
- Machete (2010) (Action|Adventure|Comedy|Crime|Thriller): user_rating=3, sim=0.222
- Cold Mountain (2003) (Drama|Romance|War): user_rating=4, sim=0.202
- King Arthur (2004) (Action|Adventure|Drama|War): user_rating=4, sim=0.188
- Helen of Troy (2003) (Action|Adventure|Drama|Romance): user_rating=4, sim=0.180
- Misérables, Les (1998) (Crime|Drama|Romance|War): user_rating=5, sim=0.175
- Forrest Gump (1994) (Comedy|Drama|Romance|War): user_rating=4, sim=0.166
- Last Samurai, The (2003) (Action|Adventure|Drama|War): user_rating=4, sim=0.161


In [7]:
# # =========================
# # 4) LLM (Mistral Instruct) — chargement & prédiction JSON (ANTI-OOM)
# # =========================
# import gc, re, json
# from typing import Dict, Tuple
# import numpy as np
# import torch
# from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig

# SYSTEM_INSTRUCTION = (
#     "You are a rating prediction assistant. "
#     "Return ONLY a valid JSON with keys rating (int 1..5), confidence (float 0..1), explanation (string). "
#     "Do not output any extra text."
# )

# def cleanup_cuda():
#     gc.collect()
#     if torch.cuda.is_available():
#         torch.cuda.empty_cache()

# def load_llm(model_name: str, device: str):
#     """
#     Anti-OOM:
#     - 4-bit NF4 quantization (VRAM hugely reduced)
#     - low_cpu_mem_usage=True
#     - return_full_text=False (avoid duplicating prompt in output)
#     """
#     cleanup_cuda()

#     tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
#     if tok.pad_token_id is None:
#         tok.pad_token = tok.eos_token

#     if device == "cuda":
#         bnb_config = BitsAndBytesConfig(
#             load_in_4bit=True,
#             bnb_4bit_quant_type="nf4",
#             bnb_4bit_compute_dtype=torch.float16,
#             bnb_4bit_use_double_quant=True,
#         )

#         model = AutoModelForCausalLM.from_pretrained(
#             model_name,
#             device_map="auto",
#             quantization_config=bnb_config,
#             low_cpu_mem_usage=True,
#         )
#     else:
#         # CPU fallback (lent mais stable)
#         model = AutoModelForCausalLM.from_pretrained(
#             model_name,
#             torch_dtype=torch.float32,
#             low_cpu_mem_usage=True,
#         )

#     gen = pipeline(
#         "text-generation",
#         model=model,
#         tokenizer=tok,
#         max_new_tokens=48,          # ↓ réduit (anti-OOM)
#         do_sample=False,
#         temperature=0.0,
#         return_full_text=False,
#         pad_token_id=tok.pad_token_id,
#         eos_token_id=tok.eos_token_id,
#     )
#     return gen, tok

# def build_prompt(user_id: int, target_movie_id: int, neigh) -> str:
#     row = movie_meta[movie_meta["movieId"] == target_movie_id].iloc[0]
#     target_desc = f"{row['title']} | {row['genres']}"

#     # ⚠️ anti-OOM: limiter le contexte RAG (évite prompts gigantesques)
#     ctx = format_rag_context(neigh.head(10) if neigh is not None else neigh)

#     return "\n".join([
#         "Task: Predict the user's rating (1..5) for the TARGET movie, given the user's past ratings.",
#         f"USER_ID: {user_id}",
#         f"TARGET_MOVIE: {target_desc}",
#         "USER_HISTORY_NEIGHBORS (most similar items from user history):",
#         ctx,
#         "Guidelines: Use past ratings as preference signal; if history empty, best guess from genres.",
#         "Output JSON only."
#     ])

# def build_mistral_inst_prompt(user_prompt: str) -> str:
#     # Instruct format correct pour Mistral
#     return f"<s>[INST] {SYSTEM_INSTRUCTION}\n\n{user_prompt}\n[/INST]"

# def parse_json(text: str) -> Dict:
#     m = re.search(r"\{.*?\}", text, flags=re.S)
#     if not m:
#         return {"rating": 3, "confidence": 0.0, "explanation": "parse_failed"}
#     try:
#         return json.loads(m.group(0))
#     except Exception:
#         return {"rating": 3, "confidence": 0.0, "explanation": "json_decode_failed"}

# @torch.inference_mode()
# def llm_predict(user_prompt: str, gen) -> Tuple[int, float, Dict]:
#     if gen is None:
#         return 3, 0.0, {"rating": 3, "confidence": 0.0, "explanation": "LLM not loaded; fallback"}

#     prompt = build_mistral_inst_prompt(user_prompt)
#     completion = gen(prompt)[0]["generated_text"]

#     obj = parse_json(completion)
#     r = int(np.clip(int(obj.get("rating", 3)), 1, 5))
#     c = float(np.clip(float(obj.get("confidence", 0.0)), 0.0, 1.0))
#     return r, c, obj

# # ---- Charge le LLM ----
# gen, tok = load_llm(LLM_MODEL_NAME, DEVICE)

# prompt_txt = build_prompt(sample_user, sample_movie, neigh)
# print(prompt_txt[:700], "...\n")
# print(llm_predict(prompt_txt, gen))


/faststorage/project/DEIC-SDU-L2-22/env/lib64/python3.9/site-packages/networkx/utils/backends.py:135: RuntimeWarning: networkx backend defined more than once: nx-loopback
  backends.update(_get_backends("networkx.backends"))


ValueError: Some modules are dispatched on the CPU or the disk. Make sure you have enough GPU RAM to fit the quantized model. If you want to dispatch the model on the CPU or the disk while keeping these modules in 32-bit, you need to set `llm_int8_enable_fp32_cpu_offload=True` and pass a custom `device_map` to `from_pretrained`. Check https://huggingface.co/docs/transformers/main/en/main_classes/quantization#offload-between-cpu-and-gpu for more details. 

In [5]:
# *********************************** Cellule 5 — LLM
# fonctionel ****** 31/12/2025
import os, gc, re, json
from typing import Dict, Tuple
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# ---- Choisir le GPU (0 ou 1) AVANT le load ----
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # mets "1" pour l'autre GPU

def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Instruction plus contraignante
SYSTEM_INSTRUCTION = """You are a rating prediction assistant.
Return ONLY a valid JSON object, no extra text, no markdown.
The JSON MUST have exactly these keys:
- "rating": integer in [1,5]
- "confidence": float in [0,1]
- "explanation": string (very short)

Example (format exactly like this):
{"rating": 4, "confidence": 0.72, "explanation": "user likes similar action/war movies"}
"""

def build_mistral_inst_prompt(user_prompt: str) -> str:
    # ✅ Force un début JSON (TRÈS efficace)
    # On met "{"" dans le prompt pour guider fortement la complétion
    return f"<s>[INST] {SYSTEM_INSTRUCTION}\n\n{user_prompt}\n[/INST]\nJSON:\n{{"

def parse_json(text: str) -> Dict:
    # Cherche le 1er objet JSON dans la sortie
    m = re.search(r"\{.*?\}", text, flags=re.S)
    if not m:
        return {"rating": 3, "confidence": 0.0, "explanation": "parse_failed"}
    try:
        return json.loads(m.group(0))
    except Exception:
        return {"rating": 3, "confidence": 0.0, "explanation": "json_decode_failed"}

def load_llm_mistral_4bit_single_gpu(model_name: str):
    cleanup_cuda()

    tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token

    # ✅ CRITIQUE: tronquer à GAUCHE pour garder la fin du prompt (dont [/INST])
    tok.truncation_side = "left"

    # Fix: certains tokenizers n'ont pas de model_max_length fiable
    if getattr(tok, "model_max_length", None) is None or tok.model_max_length > 10**6:
        tok.model_max_length = 4096

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map={"": 0},                 # tout sur le GPU visible
        quantization_config=bnb_config,
        low_cpu_mem_usage=True,
    )
    model.eval()

    # ✅ réduit la conso mémoire pendant la génération (souvent utile)
    model.config.use_cache = False

    return model, tok

@torch.inference_mode()
def llm_predict(
    user_prompt: str,
    model,
    tok,
    max_new_tokens: int = 96,
    max_input_tokens: int = 2048
) -> Tuple[int, float, Dict]:

    prompt = build_mistral_inst_prompt(user_prompt)

    inputs = tok(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=max_input_tokens,
        padding=False
    )

    input_ids = inputs["input_ids"].to(model.device)
    attn = inputs["attention_mask"].to(model.device)

    out_ids = model.generate(
        input_ids=input_ids,
        attention_mask=attn,
        max_new_tokens=max_new_tokens,
        min_new_tokens=20,          # ✅ évite sortie vide
        do_sample=False,
        temperature=0.0,
        repetition_penalty=1.05,    # ✅ réduit les boucles / tokens bizarres
        pad_token_id=tok.pad_token_id,
        eos_token_id=tok.eos_token_id,
        use_cache=False,            # ✅ cohérent avec model.config.use_cache=False
    )

    # decode uniquement la continuation
    gen_ids = out_ids[0, input_ids.shape[1]:]
    completion = tok.decode(gen_ids, skip_special_tokens=True).strip()

    # ✅ On a déjà injecté "{" dans le prompt => on reconstruit l'objet complet
    completion_full = "{" + completion

    obj = parse_json(completion_full)
    r = int(np.clip(int(obj.get("rating", 3)), 1, 5))
    c = float(np.clip(float(obj.get("confidence", 0.0)), 0.0, 1.0))
    return r, c, obj

# ---- LOAD ----
LLM_MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"
model, tok = load_llm_mistral_4bit_single_gpu(LLM_MODEL_NAME)

# ---- TEST SANITY ----
test_prompt = (
    "USER_ID: 1\n"
    "TARGET_MOVIE: Toy Story (1995) | Adventure|Animation\n"
    "USER_HISTORY_NEIGHBORS:\n"
    "- Space Jam (1996): user_rating=3, sim=0.28\n"
    "Output JSON only."
)

print(llm_predict(test_prompt, model, tok))


/faststorage/project/DEIC-SDU-L2-22/env/lib64/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/faststorage/project/DEIC-SDU-L2-22/env/lib64/python3.9/site-packages/networkx/utils/backends.py:135: RuntimeWarning: networkx backend defined more than once: nx-loopback
  backends.update(_get_backends("networkx.backends"))
Loading checkpoint shards: 100%|██████████████████████████████████████████████████████████| 3/3 [00:58<00:00, 19.63s/it]
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


(4, 0.95, {'rating': 4, 'confidence': 0.95, 'explanation': 'user has a history of liking animation films and the similarity score is high'})


In [12]:

# import os, gc, re, json
# from typing import Dict, Tuple
# import numpy as np
# import torch
# from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig

# # ---- Choisir le GPU (0 ou 1) AVANT le load ----
# os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # mets "1" si tu veux l'autre GPU

# SYSTEM_INSTRUCTION = (
#     "You are a rating prediction assistant. "
#     "Return ONLY a valid JSON with keys rating (int 1..5), confidence (float 0..1), explanation (string). "
#     "Do not output any extra text."
# )

# def cleanup_cuda():
#     gc.collect()
#     if torch.cuda.is_available():
#         torch.cuda.empty_cache()

# def load_llm_mistral_4bit_single_gpu(model_name: str):
#     cleanup_cuda()

#     tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
#     if tok.pad_token_id is None:
#         tok.pad_token = tok.eos_token

#     bnb_config = BitsAndBytesConfig(
#         load_in_4bit=True,
#         bnb_4bit_quant_type="nf4",
#         bnb_4bit_use_double_quant=True,
#         bnb_4bit_compute_dtype=torch.float16,
#     )

#     # device_map={"":0} => tout sur l’unique GPU visible (grâce à CUDA_VISIBLE_DEVICES)
#     model = AutoModelForCausalLM.from_pretrained(
#         model_name,
#         device_map={"": 0},
#         quantization_config=bnb_config,
#         low_cpu_mem_usage=True,
#     )

#     gen = pipeline(
#         "text-generation",
#         model=model,
#         tokenizer=tok,
#         max_new_tokens=64,          # réduit
#         do_sample=False,
#         temperature=0.0,
#         return_full_text=False,     # IMPORTANT
#         pad_token_id=tok.pad_token_id,
#         eos_token_id=tok.eos_token_id,
#         batch_size=1,
#     )
#     return gen, tok

# def build_mistral_inst_prompt(user_prompt: str) -> str:
#     return f"<s>[INST] {SYSTEM_INSTRUCTION}\n\n{user_prompt}\n[/INST]"

# def parse_json(text: str) -> Dict:
#     m = re.search(r"\{.*?\}", text, flags=re.S)
#     if not m:
#         return {"rating": 3, "confidence": 0.0, "explanation": "parse_failed"}
#     try:
#         return json.loads(m.group(0))
#     except Exception:
#         return {"rating": 3, "confidence": 0.0, "explanation": "json_decode_failed"}

# @torch.inference_mode()
# def llm_predict(user_prompt: str, gen) -> Tuple[int, float, Dict]:
#     prompt = build_mistral_inst_prompt(user_prompt)

#     out = gen(
#         prompt,
#         max_new_tokens=64,
#         return_full_text=False,
#         truncation=True,            # évite prompts trop longs
#     )[0]["generated_text"]

#     obj = parse_json(out)
#     r = int(np.clip(int(obj.get("rating", 3)), 1, 5))
#     c = float(np.clip(float(obj.get("confidence", 0.0)), 0.0, 1.0))
#     return r, c, obj

# # ---- Load ----
# LLM_MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"
# gen, tok = load_llm_mistral_4bit_single_gpu(LLM_MODEL_NAME)


Loading checkpoint shards: 100%|██████████████████████████████████████████████████████████| 3/3 [00:07<00:00,  2.58s/it]
Device set to use cuda:0
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [13]:
test_prompt = "USER_ID: 1\nTARGET_MOVIE: Toy Story (1995) | Adventure|Animation\nUSER_HISTORY_NEIGHBORS:\n- Space Jam (1996): user_rating=3, sim=0.28\nOutput JSON only."
print(llm_predict(test_prompt, gen))

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


(3, 0.0, {'rating': 3, 'confidence': 0.0, 'explanation': 'parse_failed'})


In [6]:
# ************************************** Cellule 6 — predict_one
def predict_one(user_id: int, movie_id: int, ratings_train: pd.DataFrame) -> Dict:
    neigh = retrieve_neighbors(user_id, movie_id, ratings_train, K_NEIGHBORS)
    user_prompt = build_prompt(user_id, movie_id, neigh)

    r_llm, c_llm, raw = llm_predict(user_prompt, model, tok)

    # ✅ éviter lam=1.0 ou 0.0
    c_llm = float(np.clip(c_llm, 0.05, 0.95))

    # tau = 1.0 + 2.0 * c_llm
    tau_min, tau_max = 0.8, 1.6
    tau = tau_min + (tau_max - tau_min) * c_llm

    grid = np.arange(1, 6)
    p_llm = np.exp(-tau * np.abs(grid - r_llm))
    p_llm = p_llm / p_llm.sum()

    p_q = grover_refine_distribution(p_llm, target_i=r_llm-1, shots=SHOTS)

    lam = 0.5 + 0.2 * c_llm   # ∈ [0.5, 0.7]
    p_f = fusion(p_llm, p_q, lam=lam)    
    # ✅ option 1 (simple): lam=c_llm
    # p_f = fusion(p_llm, p_q, lam=c_llm)
    print("p_f:", np.round(p_f, 3), "argmax p_f:", int(np.argmax(p_f)+1))

   

    # ✅ option 2 (papier, recommandée):
    # lam = 0.7 + 0.2 * c_llm
    # p_f = fusion(p_llm, p_q, lam=lam)

    r_hat = int(np.argmax(p_f) + 1)

    return {
        "userId": user_id, "movieId": movie_id,
        "rating_llm": r_llm, "confidence": c_llm,
        "rating_hat": r_hat
    }


In [35]:
# **************************************************** Cellule 7 — Test unitaire (IMPORTANT)
sample_user = int(ratings_df["userId"].iloc[0])
sample_movie = int(ratings_df["movieId"].iloc[0])

out = predict_one(sample_user, sample_movie, ratings_df)
print(out)


p_f: [0.002 0.01  0.04  0.173 0.775] argmax p_f: 5
{'userId': 1, 'movieId': 1, 'rating_llm': 5, 'confidence': 0.95, 'rating_hat': 5}


In [10]:
# # ;;;;;;;;;;;;;;;;;;;;;;;;;;;;
# import gc, re, json
# from typing import Dict, Tuple
# import numpy as np
# import torch
# from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# SYSTEM_INSTRUCTION = (
#     "You are a rating prediction assistant. "
#     "Return ONLY a valid JSON with keys rating (int 1..5), confidence (float 0..1), explanation (string). "
#     "Do not output any extra text."
# )

# def cleanup_cuda():
#     gc.collect()
#     if torch.cuda.is_available():
#         torch.cuda.empty_cache()

# def load_llm(model_name: str, device: str):
#     cleanup_cuda()

#     tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
#     if tok.pad_token_id is None:
#         tok.pad_token = tok.eos_token

#     # Anti-OOM: fp16 + low_cpu_mem_usage
#     model = AutoModelForCausalLM.from_pretrained(
#         model_name,
#         device_map="auto" if device == "cuda" else None,
#         torch_dtype=torch.float16 if device == "cuda" else torch.float32,
#         low_cpu_mem_usage=True,
#     )

#     gen = pipeline(
#         "text-generation",
#         model=model,
#         tokenizer=tok,
#         max_new_tokens=32,          # ↓↓↓ encore plus petit
#         do_sample=False,
#         temperature=0.0,
#         return_full_text=False,
#         pad_token_id=tok.pad_token_id,
#         eos_token_id=tok.eos_token_id,
#     )
#     return gen, tok

def build_prompt(user_id: int, target_movie_id: int, neigh) -> str:
    row = movie_meta[movie_meta["movieId"] == target_movie_id].iloc[0]
    target_desc = f"{row['title']} | {row['genres']}"

    # Anti-OOM: limiter RAG
    ctx = format_rag_context(neigh.head(8) if neigh is not None else neigh)

    return "\n".join([
        "Task: Predict the user's rating (1..5) for the TARGET movie, given the user's past ratings.",
        f"USER_ID: {user_id}",
        f"TARGET_MOVIE: {target_desc}",
        "USER_HISTORY_NEIGHBORS (most similar items from user history):",
        ctx,
        "Guidelines: Use past ratings as preference signal; if history empty, best guess from genres.",
        "Output JSON only."
    ])

# def build_mistral_inst_prompt(user_prompt: str) -> str:
#     return f"<s>[INST] {SYSTEM_INSTRUCTION}\n\n{user_prompt}\n[/INST]"

# def parse_json(text: str) -> Dict:
#     m = re.search(r"\{.*?\}", text, flags=re.S)
#     if not m:
#         return {"rating": 3, "confidence": 0.0, "explanation": "parse_failed"}
#     try:
#         return json.loads(m.group(0))
#     except Exception:
#         return {"rating": 3, "confidence": 0.0, "explanation": "json_decode_failed"}

# @torch.inference_mode()
# def llm_predict(user_prompt: str, gen) -> Tuple[int, float, Dict]:
#     if gen is None:
#         return 3, 0.0, {"rating": 3, "confidence": 0.0, "explanation": "LLM not loaded; fallback"}

#     prompt = build_mistral_inst_prompt(user_prompt)
#     completion = gen(prompt)[0]["generated_text"]

#     obj = parse_json(completion)
#     r = int(np.clip(int(obj.get("rating", 3)), 1, 5))
#     c = float(np.clip(float(obj.get("confidence", 0.0)), 0.0, 1.0))
#     return r, c, obj

# gen, tok = load_llm(LLM_MODEL_NAME, DEVICE)

# prompt_txt = build_prompt(sample_user, sample_movie, neigh)
# print(prompt_txt[:700], "...\n")
# print(llm_predict(prompt_txt, gen))


In [ ]:

# # =========================
# # 4) LLM (Mistral Instruct) — chargement & prédiction JSON
# # =========================
# import torch
# from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# SYSTEM_INSTRUCTION = (
#     "You are a rating prediction assistant. "
#     "Return ONLY a valid JSON with keys rating (int 1..5), confidence (float 0..1), explanation (string). "
#     "Do not output any extra text."
# )

# def load_llm(model_name: str, device: str):
#     tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
#     if tok.pad_token_id is None:
#         tok.pad_token = tok.eos_token

#     model = AutoModelForCausalLM.from_pretrained(
#         model_name,
#         device_map="auto" if device=="cuda" else None,
#         torch_dtype="auto",
#     )

#     gen = pipeline(
#         "text-generation",
#         model=model,
#         tokenizer=tok,
#         max_new_tokens=96,
#         do_sample=False,
#         temperature=0.0,
#         return_full_text=False,
#         pad_token_id=tok.pad_token_id,
#         eos_token_id=tok.eos_token_id,
#     )
#     return gen, tok

# def build_prompt(user_id: int, target_movie_id: int, neigh: pd.DataFrame) -> str:
#     row = movie_meta[movie_meta["movieId"]==target_movie_id].iloc[0]
#     target_desc = f"{row['title']} | {row['genres']}"
#     ctx = format_rag_context(neigh)
#     return "\n".join([
#         "Task: Predict the user's rating (1..5) for the TARGET movie, given the user's past ratings.",
#         f"USER_ID: {user_id}",
#         f"TARGET_MOVIE: {target_desc}",
#         "USER_HISTORY_NEIGHBORS (most similar items from user history):",
#         ctx,
#         "Guidelines: Use past ratings as preference signal; if history empty, best guess from genres.",
#         "Output JSON only."
#     ])

# def build_mistral_inst_prompt(user_prompt: str) -> str:
#     return f"<s>[INST] {SYSTEM_INSTRUCTION}\n\n{user_prompt}\n[/INST]"

# def parse_json(text: str) -> Dict:
#     m = re.search(r"\{.*?\}", text, flags=re.S)
#     if not m:
#         return {"rating": 3, "confidence": 0.0, "explanation": "parse_failed"}
#     try:
#         return json.loads(m.group(0))
#     except Exception:
#         return {"rating": 3, "confidence": 0.0, "explanation": "json_decode_failed"}

# def llm_predict(user_prompt: str, gen) -> Tuple[int, float, Dict]:
#     if gen is None:
#         return 3, 0.0, {"rating": 3, "confidence": 0.0, "explanation": "LLM not loaded; fallback"}

#     prompt = build_mistral_inst_prompt(user_prompt)
#     completion = gen(prompt)[0]["generated_text"]

#     obj = parse_json(completion)
#     r = int(np.clip(int(obj.get("rating", 3)), 1, 5))
#     c = float(np.clip(float(obj.get("confidence", 0.0)), 0.0, 1.0))
#     return r, c, obj

# # ---- Charge le LLM en décommentant ----
# # gen = None
# gen, tok = load_llm(LLM_MODEL_NAME, DEVICE)

# prompt_txt = build_prompt(sample_user, sample_movie, neigh)
# print(prompt_txt[:700], "...\n")
# print(llm_predict(prompt_txt, gen))


In [7]:

# =========================
# 5) Baseline: Quantum kNN
# =========================
import pennylane as qml
from sklearn.metrics import mean_absolute_error, mean_squared_error

def quantum_reweight_sims(sims: np.ndarray, shots: int = 400) -> np.ndarray:
    sims = np.maximum(np.array(sims, dtype=float), 0.0)
    if sims.sum() == 0:
        return np.ones_like(sims) / len(sims)

    p = sims / sims.sum()
    amps = np.sqrt(p)

    k = len(amps)
    n = int(np.ceil(np.log2(k)))
    dim = 2**n
    state = np.zeros(dim, dtype=complex)
    state[:k] = amps.astype(complex)

    target_i = int(np.argmax(p))
    dev = qml.device("default.qubit", wires=n, shots=shots)

    @qml.qnode(dev)
    def circuit():
        qml.QubitStateVector(state, wires=list(range(n)))
        qml.FlipSign(target_i, wires=list(range(n)))
        qml.GroverOperator(wires=list(range(n)))
        return qml.probs(wires=list(range(n)))

    probs = np.array(circuit()[:k], dtype=float)
    return probs / probs.sum() if probs.sum() else np.ones(k)/k

def quantum_knn_predict(user_id: int, movie_id: int, ratings_train: pd.DataFrame, k: int) -> Tuple[int, float, Dict]:
    neigh = retrieve_neighbors(user_id, movie_id, ratings_train, k)
    if neigh is None or len(neigh)==0:
        mu = float(ratings_train["rating"].mean())
        r = int(np.clip(round(mu), 1, 5))
        return r, 0.0, {"reason":"no_history"}

    sims = neigh["sim"].to_numpy(dtype=float)
    w = quantum_reweight_sims(sims, shots=SHOTS)

    r_hat = float(np.sum(w * neigh["rating"].to_numpy(dtype=float)))
    r_int = int(np.clip(round(r_hat), 1, 5))
    conf = float(np.clip(np.max(w), 0.0, 1.0))
    return r_int, conf, {"weights": w.tolist()}


In [8]:
# **************************** Cellule 4: grover_refine_distribution
# =========================
# 6) Grover refine distribution over ratings (1..5)
# =========================
def grover_refine_distribution(p_llm: np.ndarray, target_i: int, shots: int = 400, iters: int = 1) -> np.ndarray:
    p_llm = np.maximum(np.array(p_llm, dtype=float), 0.0)
    p_llm = p_llm / p_llm.sum() if p_llm.sum() else np.ones(5)/5

    amps = np.sqrt(p_llm)
    K = len(amps)     # 5
    n = int(np.ceil(np.log2(K)))  # 3
    dim = 2**n

    state = np.zeros(dim, dtype=complex)
    state[:K] = amps.astype(complex)

    dev = qml.device("default.qubit", wires=n, shots=shots)

    @qml.qnode(dev)
    def circuit():
        qml.QubitStateVector(state, wires=list(range(n)))
        for _ in range(iters):
            qml.FlipSign(int(target_i), wires=list(range(n)))
            qml.GroverOperator(wires=list(range(n)))
        return qml.probs(wires=list(range(n)))

    probs = np.array(circuit()[:K], dtype=float)
    return probs / probs.sum() if probs.sum() else np.ones(K)/K

def fusion(p_llm: np.ndarray, p_q: np.ndarray, lam: float) -> np.ndarray:
    lam = float(np.clip(lam, 0.0, 1.0))
    
    p = lam * p_llm + (1.0 - lam) * p_q
    return p / p.sum() if p.sum() else np.ones_like(p)/len(p)


In [23]:
# je veux tester lambdas = [0.25, 0.5, 0.75]
# =========================
# 7) Predictors: LLM-only, Grover-only LLM, Q-LLM (Hybrid)
# =========================
def llm_only_predict(uid: int, mid: int, ratings_train: pd.DataFrame) -> int:
    neigh = retrieve_neighbors(uid, mid, ratings_train, K_NEIGHBORS)
    prompt = build_prompt(uid, mid, neigh)
    r_llm, _, _ = llm_predict(prompt, model, tok)
    return r_llm

def grover_only_llm_predict(uid: int, mid: int, ratings_train: pd.DataFrame) -> int:
    neigh = retrieve_neighbors(uid, mid, ratings_train, K_NEIGHBORS)
    prompt = build_prompt(uid, mid, neigh)
    r_llm, c_llm, _ = llm_predict(prompt, model, tok)


    tau = 1.0 + 2.0 * c_llm
    grid = np.arange(1,6)
    p_llm = np.exp(-tau * np.abs(grid - r_llm))
    p_llm = p_llm / p_llm.sum()

    p_q = grover_refine_distribution(p_llm, target_i=r_llm-1, shots=SHOTS, iters=GROVER_ITERS)
    return int(np.argmax(p_q) + 1)

def q_llm_predict(uid: int, mid: int, ratings_train: pd.DataFrame) -> int:
    neigh = retrieve_neighbors(uid, mid, ratings_train, K_NEIGHBORS)
    prompt = build_prompt(uid, mid, neigh)
    r_llm, c_llm, _ = llm_predict(prompt, model, tok)


    tau = 1.0 + 2.0 * c_llm
    grid = np.arange(1,6)
    p_llm = np.exp(-tau * np.abs(grid - r_llm))
    p_llm = p_llm / p_llm.sum()

    p_q = grover_refine_distribution(p_llm, target_i=r_llm-1, shots=SHOTS, iters=GROVER_ITERS)
    
    # lam = 0.7 + 0.2 * c_llm  la version correcte et initiale *************************************************************************************
    lam = 0.25
    p_f = fusion(p_llm, p_q, lam=lam)
    # p_f = fusion(p_llm, p_q, lam=c_llm)
    # *** ici on prend l'argmax donc la variation de lamda n'a pas d'effet return int(np.argmax(p_f) + 1)
    grid = np.arange(1, 6)
    return float(np.sum(grid * p_f))


# quick sanity check
print("Quantum kNN:", quantum_knn_predict(sample_user, sample_movie, train_df, K_NEIGHBORS)[0])
print("LLM-only:", llm_only_predict(sample_user, sample_movie, train_df))

print("Grover-only LLM:", grover_only_llm_predict(sample_user, sample_movie, train_df))
print("Q-LLM:", q_llm_predict(sample_user, sample_movie, train_df))


Quantum kNN: 4
LLM-only: 4
Grover-only LLM: 4
Q-LLM: 3.9291192357493245


In [24]:

# =========================
# 8) Evaluation
# =========================
def eval_predictions(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae = float(mean_absolute_error(y_true, y_pred))
    acc = float(np.mean(y_true == y_pred))
    return {"RMSE": rmse, "MAE": mae, "Accuracy": acc}

def run_eval(test_df: pd.DataFrame, ratings_train: pd.DataFrame) -> Dict[str, Dict[str,float]]:
    y_true = test_df["rating"].to_numpy(dtype=int)

    preds_llm = []
    preds_qknn = []
    preds_grover = []
    preds_qllm = []

    for i, row in test_df.iterrows():
        uid = int(row["userId"]); mid = int(row["movieId"])

        r_qknn, _, _ = quantum_knn_predict(uid, mid, ratings_train, K_NEIGHBORS)
        preds_qknn.append(r_qknn)

        preds_llm.append(llm_only_predict(uid, mid, ratings_train))
        preds_grover.append(grover_only_llm_predict(uid, mid, ratings_train))
        preds_qllm.append(q_llm_predict(uid, mid, ratings_train))

        if (i+1) % 25 == 0:
            print(f"Done {i+1}/{len(test_df)}")

    out = {
        "LLM": eval_predictions(y_true, np.array(preds_llm, dtype=int)),
        "Quantum kNN": eval_predictions(y_true, np.array(preds_qknn, dtype=int)),
        "Grover-only LLM": eval_predictions(y_true, np.array(preds_grover, dtype=int)),
        "Q-LLM (Hybrid)": eval_predictions(y_true, np.array(preds_qllm, dtype=int)),
    }
    return out

results = run_eval(test_df, train_df)
results


Done 25/300
Done 50/300
Done 75/300
Done 100/300
Done 125/300
Done 150/300
Done 175/300
Done 200/300
Done 225/300
Done 250/300
Done 275/300
Done 300/300


{'LLM': {'RMSE': 1.0376254944182253,
  'MAE': 0.67,
  'Accuracy': 0.5033333333333333},
 'Quantum kNN': {'RMSE': 1.2069244660154448,
  'MAE': 0.7966666666666666,
  'Accuracy': 0.4666666666666667},
 'Grover-only LLM': {'RMSE': 1.0376254944182253,
  'MAE': 0.67,
  'Accuracy': 0.5033333333333333},
 'Q-LLM (Hybrid)': {'RMSE': 1.0519822558706333,
  'MAE': 0.8533333333333334,
  'Accuracy': 0.2733333333333333}}

In [25]:

# =========================
# 9) Résumé en tableau + export
# =========================
res_df = pd.DataFrame(results).T[["RMSE","MAE","Accuracy"]]
display(res_df)
res_df.to_csv("movielens_results_qrrp.csv", index=True)
print("Saved -> movielens_results_qrrp.csv")


,RMSE,MAE,Accuracy
LLM,1.037625,0.670000,0.503333
Quantum kNN,1.206924,0.796667,0.466667
Grover-only LLM,1.037625,0.670000,0.503333
Q-LLM (Hybrid),1.051982,0.853333,0.273333


Saved -> movielens_results_qrrp.csv
